<a href="https://colab.research.google.com/github/daviminati/Trab_Final_IA/blob/main/Sprint_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 4: Implementando um Modelo GPT do Zero

**Disciplina / Projeto Integrador LLM**

Neste notebook, construímos a arquitetura completa do modelo GPT conforme o Capítulo 4 do livro *Build a Large Language Model (From Scratch)* de Sebastian Raschka.

---

## 4.1 Configuração Inicial e Importações

Começamos importando as bibliotecas essenciais do PyTorch e verificando as versões do ambiente de execução.

In [1]:
# 1. Configuração do ambiente e importações essenciais
from importlib.metadata import version
import math
import torch
import torch.nn as nn

print("torch version:", version("torch"))

torch version: 2.11.0+cpu


## 4.2 Entregável 2: Normalização de Camada (LayerNorm)

A **Layer Normalization** ajusta as ativações de cada camada para que tenham média 0 e variância 1 ao longo da dimensão de embedding.

Diferente do BatchNorm, a LayerNorm opera individualmente em cada amostra/token, estabilizando o fluxo de gradientes durante o treinamento de redes profundas como o GPT.

Nesta célula implementamos:
* A subtração da média (`mean`) e divisão pelo desvio padrão (`sqrt(var + eps)`).
* Parâmetros aprendíveis de escala (`scale` / $\gamma$) e deslocamento (`shift` / $\beta$).

In [2]:
# Implementação da classe LayerNorm
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

# Teste básico da LayerNorm com um tensor aleatório
torch.manual_seed(123)
exemplo_batch = torch.randn(2, 3, 768) # Batch=2, Tokens=3, Dim=768
ln = LayerNorm(768)
out_ln = ln(exemplo_batch)

print("Média após LayerNorm (próxima de 0):", out_ln.mean(dim=-1).detach().numpy().round(2))
print("Variância após LayerNorm (próxima de 1):", out_ln.var(dim=-1, unbiased=False).detach().numpy().round(2))

Média após LayerNorm (próxima de 0): [[ 0.  0. -0.]
 [ 0.  0.  0.]]
Variância após LayerNorm (próxima de 1): [[1. 1. 1.]
 [1. 1. 1.]]


## 4.3 Função de Ativação GELU

Em vez da ReLU tradicional, modelos da família GPT utilizam a função de ativação **GELU** (Gaussian Error Linear Unit).

A GELU é uma função suave e não linear que pondera os valores de entrada por sua probabilidade sob uma distribuição gaussiana. Ela permite pequenos gradientes negativos para entradas abaixo de zero, o que melhora a convergência e a expressividade do modelo.

In [3]:
# Implementação da ativação GELU (Aproximação utilizada no GPT-2)
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))

# Validação do comportamento da GELU
gelu = GELU()
x_teste = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Valores de entrada: ", x_teste)
print("Saída após a GELU:", gelu(x_teste))

Valores de entrada:  tensor([-2., -1.,  0.,  1.,  2.])
Saída após a GELU: tensor([-0.0454, -0.1588,  0.0000,  0.8412,  1.9546])


## 4.4 Rede Feed-Forward (FFN)

A **Feed-Forward Network** do Transformer consiste em duas camadas lineares com a ativação GELU entre elas:
1. **Primeira projeção:** Expande a dimensão do embedding por um fator de 4 ($4 \times d_{out}$).
2. **Ativação:** Aplica a função GELU na dimensão expandida.
3. **Segunda projeção:** Projeta o tensor de volta para a dimensão original ($d_{out}$).

Essa expansão temporária permite que a rede aprenda representações e combinações de recursos mais ricas em cada posição da sequência.

In [4]:
# Implementação da subclasse FeedForward
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

# Configuração de teste para a FFN
cfg_teste = {"emb_dim": 768}
ffn = FeedForward(cfg_teste)
out_ffn = ffn(exemplo_batch)

print("Shape de entrada na FFN:", exemplo_batch.shape)
print("Shape de saída da FFN:  ", out_ffn.shape)

Shape de entrada na FFN: torch.Size([2, 3, 768])
Shape de saída da FFN:   torch.Size([2, 3, 768])


## 4.5 Módulo Multi-Head Attention

Reutilizamos aqui a classe `MultiHeadAttention` com máscara causal desenvolvida na Sprint 3. Ela permite que cada token preste atenção a tokens anteriores em múltiplos subespaços de representação em paralelo.

In [5]:
# Implementação da classe MultiHeadAttention (da Sprint 3)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out precisa ser divisível por num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)

        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

## 4.6 O Bloco Transformer (TransformerBlock)

Integrando os componentes anteriores, montamos o **TransformerBlock**.

Ele adota o padrão **Pre-LayerNorm** (LayerNorm aplicada antes dos subblocos) e inclui **Conexões Residuais (Shortcut Connections)**:
1. $x = x + \text{Dropout}(\text{Attention}(\text{LayerNorm}(x)))$
2. $x = x + \text{Dropout}(\text{FFN}(\text{LayerNorm}(x)))$

As conexões residuais evitam a atenuação/desaparecimento do gradiente durante o treinamento de arquiteturas profundas.

In [6]:
# Implementação integrada da classe TransformerBlock
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Conexão Residual 1: Bloco de Atenção
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Conexão Residual 2: Bloco FeedForward
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

# Teste unitário do TransformerBlock com a configuração GPT-2 124M
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Tamanho do vocabulário do tokenizer BPE
    "context_length": 1024, # Tamanho máximo da janela de contexto
    "emb_dim": 768,         # Dimensão do embedding do token
    "n_heads": 12,          # Número de cabeças de atenção
    "n_layers": 12,         # Quantidade de Transformer Blocks empilhados
    "drop_rate": 0.1,       # Taxa de Dropout
    "qkv_bias": False       # Desativa o viés nas projeções Q, K, V
}

torch.manual_seed(123)
block = TransformerBlock(GPT_CONFIG_124M)
out_block = block(exemplo_batch)

print("Entrada no TransformerBlock:", exemplo_batch.shape)
print("Saída do TransformerBlock:  ", out_block.shape)

Entrada no TransformerBlock: torch.Size([2, 3, 768])
Saída do TransformerBlock:   torch.Size([2, 3, 768])


## 4.7 Entregável 3: Construção da Classe GPTModel

Agora unimos todos os blocos em um modelo completo estilo GPT (`GPTModel`):
* **`tok_emb`:** Projeta Token IDs numéricos em vetores densos (`emb_dim`).
* **`pos_emb`:** Adiciona embeddings posicionais aprendíveis para indicar a ordem dos tokens.
* **`trf_blocks`:** Sequência empilhada de $N$ `TransformerBlock`s (`n_layers`).
* **`final_norm`:** Normalização final aplicada antes do cabeçote de saída.
* **`out_head`:** Camada linear (LM Head) que converte as representações ocultas finais em **Logits** não normalizados para cada palavra do vocabulário.

In [7]:
# Implementação da classe GPTModel
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Empilhamento de N blocos Transformer via nn.Sequential
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# Instanciação e teste da passagem direta (forward)
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

# Lote simulado de 2 sequências de 4 tokens cada
batch_ids = torch.tensor([[16833, 3626, 6100, 344], [40, 1107, 588, 287]])
logits = model(batch_ids)

print("Formato do tensor de entrada (Token IDs):", batch_ids.shape)
print("Formato dos Logits de saída (Batch, Seq_Len, Vocab_Size):", logits.shape)

Formato do tensor de entrada (Token IDs): torch.Size([2, 4])
Formato dos Logits de saída (Batch, Seq_Len, Vocab_Size): torch.Size([2, 4, 50257])


## 4.8 Entregável 4: Geração Autorregressiva de Texto

Implementamos a função `generate_text_simple` para realizar a inferência do modelo passo a passo.

O processo funciona da seguinte forma:
1. Recebe uma sequência inicial de tokens (prompt).
2. Garante que a entrada não exceda o limite da janela de contexto (`context_size`).
3. Executa a passagem direta no modelo sem calcular gradientes (`torch.no_grad()`).
4. Extrai os logits do **último token** gerado.
5. Seleciona o próximo token usando amostragem gulosa (`torch.argmax`).
6. Concatena o novo token à sequência e repete o processo até atingir `max_new_tokens`.

In [8]:
# Função para geração de texto autorregressiva simples (Greedy Decoding)
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        # Trunca o contexto se ele for maior do que o suportado pelo modelo
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        # Pega os logits referentes à última posição da sequência
        logits = logits[:, -1, :]

        # Seleção gulosa: escolhe o token com maior valor de logit
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # Concatena o novo token ao histórico
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

# Teste da função de geração
torch.manual_seed(123)

# Prompt simulado representado por Token IDs
prompt_ids = torch.tensor([[15496, 11, 314, 716]]) # Exemplo: "Hello, I am"

out_ids = generate_text_simple(
    model=model,
    idx=prompt_ids,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Sequência de Token IDs gerada:", out_ids)
print("Comprimento total da sequência:", out_ids.shape[1])

Sequência de Token IDs gerada: tensor([[15496,    11,   314,   716, 40179, 49958, 43151,   334, 21945, 33043]])
Comprimento total da sequência: 10


## 4.9 Entregável 5: Experimentação e Análise de Parâmetros

Nesta seção realizamos o experimento comparativo de portes da arquitetura. Analisamos como a variação da dimensão dos embeddings (`emb_dim`), do número de camadas (`n_layers`) e de cabeças (`n_heads`) altera o **total de parâmetros treináveis** do modelo e o consumo estimado de memória.

In [9]:
# Função auxiliar para contagem de parâmetros treináveis
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Experimento 1: Comparação de diferentes portes de arquitetura do GPT
configs_exp = {
    "GPT-Small (Prototipagem)": {
        "vocab_size": 50257, "context_length": 256, "emb_dim": 384,
        "n_heads": 6, "n_layers": 6, "drop_rate": 0.1, "qkv_bias": False
    },
    "GPT-2 Base (124M)": GPT_CONFIG_124M,
    "GPT-2 Medium (Simulado)": {
        "vocab_size": 50257, "context_length": 1024, "emb_dim": 1024,
        "n_heads": 16, "n_layers": 24, "drop_rate": 0.1, "qkv_bias": False
    }
}

print("=== Experimento 1: Comparação de Tamanho das Arquiteturas ===")
for nome, cfg in configs_exp.items():
    modelo_exp = GPTModel(cfg)
    total_params = count_parameters(modelo_exp)
    mem_mb = (total_params * 4) / (1024 ** 2) # Estimativa de memória em float32

    print(f"\nModelo: {nome}")
    print(f" - Dimensão de Embedding (emb_dim): {cfg['emb_dim']}")
    print(f" - Quantidade de Camadas (n_layers): {cfg['n_layers']}")
    print(f" - Cabeças de Atenção (n_heads):   {cfg['n_heads']}")
    print(f" - Parâmetros Treináveis:          {total_params:,}")
    print(f" - Memória Estimada dos Pesos:     ~{mem_mb:.2f} MB")

# Experimento 2: Efeito da variação da dimensão da janela de contexto (context_length)
print("\n=== Experimento 2: Impacto da Janela de Contexto no GPT-2 Base ===")
for ctx_len in [256, 512, 1024, 2048]:
    cfg_temp = GPT_CONFIG_124M.copy()
    cfg_temp["context_length"] = ctx_len
    m_temp = GPTModel(cfg_temp)
    print(f"Janela de Contexto: {ctx_len:4d} tokens | Parâmetros de Positional Embedding: {m_temp.pos_emb.weight.numel():,} | Total Modelo: {count_parameters(m_temp):,}")

=== Experimento 1: Comparação de Tamanho das Arquiteturas ===

Modelo: GPT-Small (Prototipagem)
 - Dimensão de Embedding (emb_dim): 384
 - Quantidade de Camadas (n_layers): 6
 - Cabeças de Atenção (n_heads):   6
 - Parâmetros Treináveis:          49,336,320
 - Memória Estimada dos Pesos:     ~188.20 MB

Modelo: GPT-2 Base (124M)
 - Dimensão de Embedding (emb_dim): 768
 - Quantidade de Camadas (n_layers): 12
 - Cabeças de Atenção (n_heads):   12
 - Parâmetros Treináveis:          163,009,536
 - Memória Estimada dos Pesos:     ~621.83 MB

Modelo: GPT-2 Medium (Simulado)
 - Dimensão de Embedding (emb_dim): 1024
 - Quantidade de Camadas (n_layers): 24
 - Cabeças de Atenção (n_heads):   16
 - Parâmetros Treináveis:          406,212,608
 - Memória Estimada dos Pesos:     ~1549.58 MB

=== Experimento 2: Impacto da Janela de Contexto no GPT-2 Base ===
Janela de Contexto:  256 tokens | Parâmetros de Positional Embedding: 196,608 | Total Modelo: 162,419,712
Janela de Contexto:  512 tokens | Parâ